# Entity Comparison API — manual test

Posts location master data + an entity payload, then inspects what the dynamic rule
engine wrote to `validation_results`. Validation now runs against the **join** of
`entities` and `entities_location` on `(systemCode, businessEntityCode)`.

**Before running:**
1. `python seed_validation_rules.py`  (one-time, loads the rule set)
2. `python run.py`  (starts the API on http://localhost:8000)
3. Run `reset_db.ipynb` for a clean slate (entities skip on re-POST and are **not**
   re-validated, and `entities_location` has no de-dup).

**Order matters:** locations must be POSTed **before** the entities, because the
join happens synchronously while each entity is inserted. Posting entities first
means the rule engine sees no `Country` and every location-based rule passes by
default.

Test data files: `sample_locations.json`, `sample_payload.json` — edit those to
change the inputs.

In [ ]:
# pip install requests pandas  # uncomment if needed
import os
import json
import requests
import pandas as pd

BASE_URL = "http://localhost:8000"

# resolve the project root (folder containing run.py) so relative paths work
# no matter where the kernel was started
_here = os.getcwd()
while not os.path.exists(os.path.join(_here, "run.py")) and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
PROJECT_ROOT = _here

LOCATIONS_FILE = os.path.join(PROJECT_ROOT, "sample_locations.json")
PAYLOAD_FILE = os.path.join(PROJECT_ROOT, "sample_payload.json")

print("project root:", PROJECT_ROOT)
print("liveness    :", requests.get(f"{BASE_URL}/docs").status_code)

## 1. POST location master data  (run this first)

In [ ]:
def post_locations(locations, base_url=BASE_URL):
    """POST a list of location master-data records to /entities/inputlocation.

    Each item must carry `systemCode`, `Code` (-> businessEntityCode) and
    `Country`; `Name`, `Type`, `Alternate_code`, `City`, `Zip` are optional.
    Returns the endpoint's per-row summary list.
    """
    if isinstance(locations, dict):
        locations = [locations]
    resp = requests.post(f"{base_url}/entities/inputlocation", json=locations)
    resp.raise_for_status()
    return resp.json()


with open(LOCATIONS_FILE, "r", encoding="utf-8") as f:
    locations = json.load(f)

print(f"loaded {len(locations)} location(s) from {LOCATIONS_FILE}")
location_result = post_locations(locations)
print(json.dumps(location_result, indent=2))

In [ ]:
# confirm what landed in entities_location
locs = requests.get(f"{BASE_URL}/entities/locations").json()
pd.DataFrame(locs)

## 2. POST the entity payload

In [ ]:
with open(PAYLOAD_FILE, "r", encoding="utf-8") as f:
    payload = json.load(f)

mappings = payload.get("mappingsInformation", [payload])
print(f"loaded {len(mappings)} mapping(s) from {PAYLOAD_FILE}")
pd.DataFrame(mappings)

In [ ]:
resp = requests.post(f"{BASE_URL}/entities/process", json=payload)
print(resp.status_code)
process_result = resp.json()
print(json.dumps(process_result, indent=2))

## 3. GET /entities  (what got stored)

In [ ]:
entities = requests.get(f"{BASE_URL}/entities").json()
pd.DataFrame(entities)

## 4. GET /entities/validation-results  (entity ⋈ location)

In [ ]:
vr = requests.get(f"{BASE_URL}/entities/validation-results").json()
vr_df = pd.DataFrame(vr)
vr_df

In [ ]:
# pass/fail matrix: one row per entity, one column per rule
if not vr_df.empty:
    matrix = vr_df.pivot_table(
        index="businessEntityCode",
        columns="rule_id",
        values="passed",
        aggfunc="last",
    )
    display(matrix.replace({1: "PASS", 0: "FAIL", True: "PASS", False: "FAIL"}))

    print("\nFailures only:")
    display(vr_df[vr_df["passed"].isin([0, False])][
        ["businessEntityCode", "rule_id", "severity", "status", "description"]
    ])

## What to expect

**The join is always applied:** every entity is validated against
`entity_json` **merged with** its `entities_location` row on
`(systemCode, businessEntityCode)`. `get /entities/locations` (cell 1) shows what
was joined in; a missing location row just leaves `Country` / `City` / … blank.

### With the shipped files + the current `seed_validation_rules.py`

Active rules: `VAL-UK-Field`, `FIELD-EXIST`, `FIELD-EOID-Unique`. All three ultimately
test `UKEOID` / `UKFID` / `EOID`, which every row in `sample_payload.json` already
has — so the only failure is the duplicate `EOID`:

| Entity | Joined Country | VAL-UK-Field | FIELD-EXIST | FIELD-EOID-Unique |
|---|---|---|---|---|
| `SGLN_mapping` | `SG` | PASS | PASS | PASS |
| `DOCS99` | `MY` | PASS | PASS | PASS |
| `BG01` | `BG` | PASS | PASS | PASS |
| `BG02` | `DE` | PASS | PASS | PASS |
| `BGN1` | `JP` | PASS | PASS | **FAIL** — `EOID LEBGR1e003Tp4N3J` duplicates `BG01` (`status = -1`) |

`BG01` / `BG02` do exercise the `Country starts_with EU-list` branch (their country
codes are in the list), but the `THEN` check passes too, so the result is still PASS.

### Seeing the join change a result

With this ruleset the join is wired but **invisible** — no active rule depends on a
location-only field. To watch it flip PASS/FAIL, add a rule that checks one, e.g. a
Data-Type `Exist` check on `City`: it fails when locations aren't loaded and passes
once they are.

### Other notes

- `passed = 1/true` also means "rule not applicable" (a `Trigger` / `IF` was false).
- `status` / `description` are `NaN` on every passing row; only populated on failure.
- Re-POSTing the same payload → entities come back `skipped / already_exists` and
  **no new `validation_results` rows**. Run `reset_db.ipynb` between full runs.
- Locations must be POSTed **before** the entities (the join runs during insert).